In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox

# =========================================================
# 1. CARGA
# =========================================================
path = "../data/interim/dataset_clean.csv"
df = pd.read_csv(path)

print("Dataset cargado:", df.shape)

# =========================================================
# 2. TARGET Y SPLIT
# =========================================================
target = "Revenue"

X = df.drop(columns=[target]).copy()
y = df[target].copy()

# convertir target a binario si viene como texto o bool
if y.dtype == "object":
    y = y.astype(str).str.strip().str.lower().map({
        "true": 1, "false": 0, "yes": 1, "no": 0
    })
elif str(y.dtype) == "bool":
    y = y.astype(int)

# eliminar filas con target nulo
mask = y.notna()
X = X.loc[mask].copy()
y = y.loc[mask].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Split:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

# =========================================================
# 3. HELPERS
# =========================================================
feature_docs_list = []

def document_feature(nombre, calculo, utilidad):
    feature_docs_list.append({
        "feature": nombre,
        "como_se_calcula": calculo,
        "por_que_podria_ser_util": utilidad
    })

def safe_divide(a, b):
    out = a / b
    out = out.replace([np.inf, -np.inf], np.nan)
    return out

def clean_bool_col(series):
    if str(series.dtype) == "bool":
        return series.astype(int)
    if series.dtype == "object":
        mapped = series.astype(str).str.strip().str.lower().map({
            "true": 1, "false": 0, "yes": 1, "no": 0
        })
        if mapped.notna().sum() > 0:
            return mapped
    return series

def add_feature_if_possible(df, name, expr_func, calc_desc, util_desc):
    try:
        new_col = expr_func(df)
        if new_col is not None:
            df[name] = new_col
            document_feature(name, calc_desc, util_desc)
    except Exception:
        pass

def add_features(df):
    df = df.copy()

    # -------------------------
    # Normalización básica
    # -------------------------
    if "Weekend" in df.columns:
        df["Weekend"] = clean_bool_col(df["Weekend"])

    # -------------------------
    # 1) FEATURES DERIVADAS
    # Ratios, diferencias, agregaciones
    # -------------------------
    add_feature_if_possible(
        df,
        "Admin_Duration_per_Page",
        lambda d: safe_divide(d["Administrative_Duration"], d["Administrative"] + 1),
        "Administrative_Duration / (Administrative + 1)",
        "Mide el tiempo medio por página administrativa."
    )

    add_feature_if_possible(
        df,
        "Info_Duration_per_Page",
        lambda d: safe_divide(d["Informational_Duration"], d["Informational"] + 1),
        "Informational_Duration / (Informational + 1)",
        "Resume cuánto tiempo dedica a páginas informativas."
    )

    add_feature_if_possible(
        df,
        "Product_Duration_per_Page",
        lambda d: safe_divide(d["ProductRelated_Duration"], d["ProductRelated"] + 1),
        "ProductRelated_Duration / (ProductRelated + 1)",
        "Captura profundidad de exploración de páginas de producto."
    )

    add_feature_if_possible(
        df,
        "Total_PageTypes",
        lambda d: d["Administrative"].fillna(0) + d["Informational"].fillna(0) + d["ProductRelated"].fillna(0),
        "Administrative + Informational + ProductRelated",
        "Mide el volumen total de páginas visitadas."
    )

    add_feature_if_possible(
        df,
        "Total_Duration",
        lambda d: d["Administrative_Duration"].fillna(0) + d["Informational_Duration"].fillna(0) + d["ProductRelated_Duration"].fillna(0),
        "Administrative_Duration + Informational_Duration + ProductRelated_Duration",
        "Resume el tiempo total de navegación del usuario."
    )

    add_feature_if_possible(
        df,
        "Avg_Duration_per_Page",
        lambda d: safe_divide(d["Total_Duration"], d["Total_PageTypes"] + 1),
        "Total_Duration / (Total_PageTypes + 1)",
        "Aproxima el tiempo promedio por página."
    )

    add_feature_if_possible(
        df,
        "Product_minus_Admin",
        lambda d: d["ProductRelated"].fillna(0) - d["Administrative"].fillna(0),
        "ProductRelated - Administrative",
        "Mide si el usuario se enfoca más en productos que en contenido administrativo."
    )

    add_feature_if_possible(
        df,
        "Exit_minus_Bounce",
        lambda d: d["ExitRates"].fillna(0) - d["BounceRates"].fillna(0),
        "ExitRates - BounceRates",
        "Captura diferencia entre abandono y rebote."
    )

    # -------------------------
    # 2) EXTRACCIÓN DE FECHAS
    # (si existe una columna de fecha)
    # -------------------------
    possible_date_cols = [c for c in df.columns if "date" in c.lower() or "fecha" in c.lower()]
    if len(possible_date_cols) > 0:
        date_col = possible_date_cols[0]
        dt = pd.to_datetime(df[date_col], errors="coerce")

        df["anio"] = dt.dt.year
        df["mes_fecha"] = dt.dt.month
        df["dia"] = dt.dt.day
        df["dow"] = dt.dt.dayofweek

        document_feature("anio", f"Año extraído de {date_col}", "Puede capturar tendencias anuales.")
        document_feature("mes_fecha", f"Mes extraído de {date_col}", "Puede capturar estacionalidad mensual.")
        document_feature("dia", f"Día del mes extraído de {date_col}", "Puede detectar patrones intra-mes.")
        document_feature("dow", f"Día de la semana extraído de {date_col}", "Puede capturar comportamiento por día de semana.")

    # -------------------------
    # 3) INTERACCIONES
    # -------------------------
    add_feature_if_possible(
        df,
        "PageValues_x_ProductRelated",
        lambda d: d["PageValues"] * d["ProductRelated"],
        "PageValues * ProductRelated",
        "Combina valor comercial con cantidad de páginas de producto."
    )

    add_feature_if_possible(
        df,
        "Exit_x_Bounce",
        lambda d: d["ExitRates"] * d["BounceRates"],
        "ExitRates * BounceRates",
        "Refuerza señales de abandono o mala retención."
    )

    add_feature_if_possible(
        df,
        "Admin_x_Info",
        lambda d: d["Administrative"] * d["Informational"],
        "Administrative * Informational",
        "Captura interacción entre exploración administrativa e informativa."
    )

    add_feature_if_possible(
        df,
        "Duration_x_PageValues",
        lambda d: d["Total_Duration"] * d["PageValues"],
        "Total_Duration * PageValues",
        "Combina intensidad de navegación con valor comercial."
    )

    # -------------------------
    # 4) FLAGS ÚTILES
    # -------------------------
    if "VisitorType" in df.columns:
        df["IsReturningVisitor"] = (
            df["VisitorType"].astype(str).str.strip().str.lower() == "returning_visitor"
        ).astype(int)
        document_feature(
            "IsReturningVisitor",
            "1 si VisitorType == 'Returning_Visitor', 0 en otro caso",
            "Distingue usuarios recurrentes de nuevos."
        )

    if "SpecialDay" in df.columns:
        df["IsSpecialDay"] = (df["SpecialDay"].fillna(0) > 0).astype(int)
        document_feature(
            "IsSpecialDay",
            "1 si SpecialDay > 0, 0 en otro caso",
            "Captura cercanía a fechas especiales."
        )

    if "Weekend" in df.columns:
        if str(df["Weekend"].dtype) in ["int64", "float64", "Int64"]:
            df["IsWeekend"] = df["Weekend"].fillna(0)
        else:
            df["IsWeekend"] = 0
        document_feature(
            "IsWeekend",
            "Conversión de Weekend a indicador binario",
            "Separa sesiones de fin de semana de días laborables."
        )

    # -------------------------
    # 5) TRANSFORMACIÓN DE MONTH
    # -------------------------
    if "Month" in df.columns:
        month_map = {
            "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5,
            "jun": 6, "june": 6, "jul": 7, "aug": 8, "sep": 9,
            "oct": 10, "nov": 11, "dec": 12
        }

        month_clean = df["Month"].astype(str).str.strip().str.lower()
        df["Month_num"] = month_clean.map(month_map)

        if df["Month_num"].notna().sum() > 0:
            moda_mes = df["Month_num"].mode().iloc[0]
            df["Month_num"] = df["Month_num"].fillna(moda_mes)

        df["Month_sin"] = np.sin(2 * np.pi * df["Month_num"] / 12)
        df["Month_cos"] = np.cos(2 * np.pi * df["Month_num"] / 12)

        document_feature("Month_num", "Mapeo de Month a 1..12", "Representa el mes de forma ordenada.")
        document_feature("Month_sin", "sin(2*pi*Month_num/12)", "Captura estacionalidad cíclica.")
        document_feature("Month_cos", "cos(2*pi*Month_num/12)", "Complementa la estacionalidad cíclica.")

    # -------------------------
    # 6) BINNING DE VARIABLES NUMÉRICAS
    # -------------------------
    if "ProductRelated" in df.columns:
        df["ProductRelated_bin"] = pd.cut(
            df["ProductRelated"],
            bins=[-1, 10, 50, 200, np.inf],
            labels=["low", "medium", "high", "very_high"]
        ).astype(str)
        document_feature(
            "ProductRelated_bin",
            "Binning de ProductRelated en low/medium/high/very_high",
            "Agrupa intensidad de exploración de productos."
        )

    if "PageValues" in df.columns:
        df["PageValues_bin"] = pd.cut(
            df["PageValues"],
            bins=[-0.01, 0, 20, 100, np.inf],
            labels=["zero", "low", "medium", "high"]
        ).astype(str)
        document_feature(
            "PageValues_bin",
            "Binning de PageValues en zero/low/medium/high",
            "Resume rangos de valor comercial."
        )

    if "Total_Duration" in df.columns:
        try:
            df["Total_Duration_bin"] = pd.qcut(
                df["Total_Duration"],
                q=4,
                labels=["Q1", "Q2", "Q3", "Q4"],
                duplicates="drop"
            ).astype(str)
            document_feature(
                "Total_Duration_bin",
                "Cuartiles de Total_Duration",
                "Agrupa sesiones por intensidad de tiempo."
            )
        except Exception:
            pass

    # limpieza final
    df = df.replace([np.inf, -np.inf], np.nan)

    return df

# =========================================================
# 4. FEATURE ENGINEERING
# =========================================================
X_train = add_features(X_train)
X_test = add_features(X_test)

print("Features creadas:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

# =========================================================
# 5. IMPUTACIÓN SIMPLE
# =========================================================
num_cols_base = X_train.select_dtypes(include=np.number).columns.tolist()
for col in num_cols_base:
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

cat_cols_base = X_train.select_dtypes(exclude=np.number).columns.tolist()
for col in cat_cols_base:
    moda = X_train[col].mode().iloc[0] if X_train[col].notna().sum() > 0 else "missing"
    X_train[col] = X_train[col].fillna(moda)
    X_test[col] = X_test[col].fillna(moda)

# =========================================================
# 6. TRANSFORMACIONES: LOG, BOX-COX, YEO-JOHNSON
# =========================================================
log_candidates = [
    "Administrative_Duration",
    "Informational_Duration",
    "ProductRelated_Duration",
    "PageValues",
    "ProductRelated",
    "Administrative",
    "Informational",
    "Total_Duration",
    "Total_PageTypes"
]

log_applied = []
for col in log_candidates:
    if col in X_train.columns:
        if (X_train[col] >= 0).all() and (X_test[col] >= 0).all():
            X_train[f"{col}_log1p"] = np.log1p(X_train[col])
            X_test[f"{col}_log1p"] = np.log1p(X_test[col])
            log_applied.append(col)
            document_feature(
                f"{col}_log1p",
                f"log1p({col})",
                "Reduce asimetría en variables no negativas."
            )

boxcox_candidates = [
    "Administrative_Duration",
    "Informational_Duration",
    "ProductRelated_Duration",
    "PageValues",
    "Total_Duration"
]

boxcox_applied = {}
for col in boxcox_candidates:
    if col in X_train.columns:
        if (X_train[col] > 0).all() and (X_test[col] > 0).all():
            transformed_train, lam = boxcox(X_train[col])
            X_train[f"{col}_boxcox"] = transformed_train
            if lam == 0:
                X_test[f"{col}_boxcox"] = np.log(X_test[col])
            else:
                X_test[f"{col}_boxcox"] = (np.power(X_test[col], lam) - 1) / lam
            boxcox_applied[col] = lam
            document_feature(
                f"{col}_boxcox",
                f"Box-Cox({col}) con lambda={round(lam, 4)}",
                "Reduce asimetría en variables estrictamente positivas."
            )

# Yeo-Johnson
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
X_train[num_cols] = X_train[num_cols].replace([np.inf, -np.inf], np.nan)
X_test[num_cols] = X_test[num_cols].replace([np.inf, -np.inf], np.nan)

for col in num_cols:
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

yj_cols = []
for col in num_cols:
    if np.isfinite(X_train[col]).all() and np.isfinite(X_test[col]).all():
        yj_cols.append(col)

pt = PowerTransformer(method="yeo-johnson", standardize=False)
X_train_yj = pt.fit_transform(X_train[yj_cols])
X_test_yj = pt.transform(X_test[yj_cols])

yj_feature_names = [f"{c}_yeojohnson" for c in yj_cols]
X_train[yj_feature_names] = X_train_yj
X_test[yj_feature_names] = X_test_yj

for c in yj_cols:
    document_feature(
        f"{c}_yeojohnson",
        f"Yeo-Johnson({c})",
        "Reduce asimetría permitiendo ceros y valores no estrictamente positivos."
    )

print("Log aplicado a:", log_applied)
print("Box-Cox aplicado a:", list(boxcox_applied.keys()))
print("Yeo-Johnson aplicado a", len(yj_cols), "columnas")

# =========================================================
# 7. SELECCIÓN PRELIMINAR
# - varianza nula
# - redundancia por alta correlación
# =========================================================

# 7.1 eliminar varianza nula
numeric_cols_before = X_train.select_dtypes(include=np.number).columns.tolist()
zero_var_cols = [col for col in numeric_cols_before if X_train[col].nunique(dropna=False) <= 1]

if len(zero_var_cols) > 0:
    X_train = X_train.drop(columns=zero_var_cols, errors="ignore")
    X_test = X_test.drop(columns=zero_var_cols, errors="ignore")

print("Columnas eliminadas por varianza nula:", zero_var_cols)

# 7.2 eliminar alta correlación
numeric_cols_after = X_train.select_dtypes(include=np.number).columns.tolist()
corr_matrix = X_train[numeric_cols_after].corr().abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [column for column in upper.columns if any(upper[column] > 0.95)]

if len(high_corr_cols) > 0:
    X_train = X_train.drop(columns=high_corr_cols, errors="ignore")
    X_test = X_test.drop(columns=high_corr_cols, errors="ignore")

print("Columnas eliminadas por alta correlación (>0.95):", high_corr_cols)

# =========================================================
# 8. DOCUMENTACIÓN FINAL
# =========================================================
feature_docs = pd.DataFrame(feature_docs_list).drop_duplicates(subset=["feature"]).reset_index(drop=True)

# =========================================================
# 9. GUARDAR RESULTADOS
# =========================================================
output = Path("../data/processed")
output.mkdir(parents=True, exist_ok=True)

X_train.to_csv(output / "X_train_fe.csv", index=False)
X_test.to_csv(output / "X_test_fe.csv", index=False)
y_train.to_csv(output / "y_train.csv", index=False)
y_test.to_csv(output / "y_test.csv", index=False)
feature_docs.to_csv(output / "feature_docs.csv", index=False)

# =========================================================
# 10. RESULTADO FINAL
# =========================================================
print("\nArchivos guardados en:", output.resolve())
print("FINAL SHAPES:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nResumen:")
print("Nuevas features documentadas:", feature_docs.shape[0])
print("Varianza nula eliminadas:", len(zero_var_cols))
print("Alta correlación eliminadas:", len(high_corr_cols))

display(feature_docs.head(20))
display(X_train.head())

Dataset cargado: (12205, 18)
Split:
X_train: (9764, 17)
X_test : (2441, 17)
Features creadas:
X_train: (9764, 38)
X_test : (2441, 38)
Log aplicado a: []
Box-Cox aplicado a: []
Yeo-Johnson aplicado a 32 columnas
Columnas eliminadas por varianza nula: ['IsReturningVisitor', 'Month_num', 'Month_sin', 'Month_cos', 'IsReturningVisitor_yeojohnson']
Columnas eliminadas por alta correlación (>0.95): ['IsWeekend', 'ProductRelated_Duration_yeojohnson', 'SpecialDay_yeojohnson', 'Month_yeojohnson', 'OperatingSystems_yeojohnson', 'Region_yeojohnson', 'VisitorType_yeojohnson', 'Weekend_yeojohnson', 'Admin_Duration_per_Page_yeojohnson', 'Info_Duration_per_Page_yeojohnson', 'Product_Duration_per_Page_yeojohnson', 'Avg_Duration_per_Page_yeojohnson', 'Product_minus_Admin_yeojohnson', 'Duration_x_PageValues_yeojohnson', 'IsSpecialDay_yeojohnson', 'IsWeekend_yeojohnson']

Archivos guardados en: C:\Users\Admin\dp261-g4\data\processed
FINAL SHAPES:
X_train: (9764, 49)
X_test : (2441, 49)
y_train: (9764,)
y_

,feature,como_se_calcula,por_que_podria_ser_util
0,Admin_Duration_per_Page,Administrative_Duration / (Administrative + 1),Mide el tiempo medio por página administrativa.
1,Info_Duration_per_Page,Informational_Duration / (Informational + 1),Resume cuánto tiempo dedica a páginas informat...
2,Product_Duration_per_Page,ProductRelated_Duration / (ProductRelated + 1),Captura profundidad de exploración de páginas ...
3,Total_PageTypes,Administrative + Informational + ProductRelated,Mide el volumen total de páginas visitadas.
4,Total_Duration,Administrative_Duration + Informational_Durati...,Resume el tiempo total de navegación del usuario.
5,Avg_Duration_per_Page,Total_Duration / (Total_PageTypes + 1),Aproxima el tiempo promedio por página.
6,Product_minus_Admin,ProductRelated - Administrative,Mide si el usuario se enfoca más en productos ...
7,Exit_minus_Bounce,ExitRates - BounceRates,Captura diferencia entre abandono y rebote.
8,PageValues_x_ProductRelated,PageValues * ProductRelated,Combina valor comercial con cantidad de página...
9,Exit_x_Bounce,ExitRates * BounceRates,Refuerza señales de abandono o mala retención.


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,ExitRates_yeojohnson,PageValues_yeojohnson,Browser_yeojohnson,TrafficType_yeojohnson,Total_PageTypes_yeojohnson,Total_Duration_yeojohnson,Exit_minus_Bounce_yeojohnson,PageValues_x_ProductRelated_yeojohnson,Exit_x_Bounce_yeojohnson,Admin_x_Info_yeojohnson
7469,1.699884,0.687123,0.385143,-0.246257,0.223233,-0.080532,-0.450137,-0.799816,1.683796,-0.310240,...,-1.417496,0.253611,-0.244537,-0.783864,1.273795,0.297847,-0.450272,0.353544,0.271056,0.539946
11804,-0.702302,-0.460019,-0.398824,-0.246257,-0.045875,-0.159255,-0.450137,-0.739421,1.511767,-0.310240,...,-1.261958,0.251933,-0.244537,-0.783864,-1.718344,-1.344281,-0.357458,-0.070224,0.255321,0.255463
3250,-0.702302,-0.460019,-0.398824,-0.246257,-0.427111,-0.409381,-0.450137,-0.287267,-0.318962,3.696611,...,-0.359855,-0.693092,-1.350429,-0.783864,-2.534339,-1.921038,0.144804,0.133014,0.115611,0.255463
204,-0.101756,-0.234648,-0.398824,-0.246257,-0.606516,-0.337835,-0.450137,-0.279331,-0.318962,-0.310240,...,-0.347857,-0.693092,-0.244537,-0.783864,-1.639852,-1.246086,0.151051,0.187115,0.112747,0.039998
4392,-0.702302,-0.460019,-0.398824,-0.246257,-0.359834,0.204365,-0.450137,-0.609440,-0.318962,-0.310240,...,-0.956456,-0.693092,-0.244537,0.768991,-2.381686,-0.659732,-0.179530,0.112468,0.219369,0.255463
